# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will print an overview of each record set (`@id`) and list the fields and column IDs included for each. All entity references use their canonical `@id`.

In [ ]:
# List all RecordSets in the dataset
recordsets = list(dataset.metadata.record_sets)
print("Available Record Sets:")
for rs in recordsets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, Name: {field.name}, DataType: {field.data_type}")
    print("  Columns:")
    for column in rs.columns:
        print(f"    - Column @id: {column.id}, Name: {column.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We select the main protein quantification table for demonstration (`cr:RecordSet/protein-quantitation`), but you may select other record sets by their `@id` from above.

In [ ]:
# List all record set @ids for processing
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

print("Loading data from available record sets:")
for record_set_id in record_set_ids:
    print(f"- Processing {record_set_id}")
    try:
        # Each yields a dict per record, indexed by field @ids
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Loaded {len(df)} records.")
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error loading: {e}")

# For demonstration, pick a main record set for further analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Fields in {main_record_set_id}:")
    print(list(dataframes[main_record_set_id].columns))
    dataframes[main_record_set_id].head()
else:
    print("No record sets available for DataFrame extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

We will:
- Select a numeric field (e.g., coverage percentage `cr:Field/coverage_percent` or peptide count `cr:Field/peptide_count`),
- Filter records where the value exceeds a threshold,
- Normalize the values of the selected field,
- Group data by protein description.

Modify the field `@id`s as needed based on the output above.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]
    
    # Example field ids (change to your dataset's actual @ids from printout above)
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        # Heuristic: use a likely coverage or count field
        if 'coverage' in col:
            numeric_field_id = col
        if group_field_id is None and ('description' in col or 'protein' in col):
            group_field_id = col
    
    # If not found, just use the first available numeric field
    if numeric_field_id is None:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    print(f"Using numeric field: {numeric_field_id}")
    print(f"Grouping field: {group_field_id}")
    
    threshold = 10
    # Coerce to numeric if required
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field and show mean
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot the distribution of the selected numeric field, and a bar chart of group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    # Histogram
    sns.histplot(df[numeric_field_id].dropna(), ax=ax[0], kde=True)
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    
    # Group mean bar chart (if group field exists)
    if group_field_id and group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
        sns.barplot(y=group_means.index, x=group_means.values, ax=ax[1])
        ax[1].set_title(f"Top 10 groups by mean {numeric_field_id}")
        ax[1].set_xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, explore, and analyze the [FAIR^2 Mass Spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

Key findings and steps included:
- Listing record sets and field structure via Croissant `@id` references
- Loading tabular protein data into pandas frames
- Filtering, normalizing, and grouping protein quantification results
- Visualizing data distributions and summary statistics

**You can extend this notebook by exploring additional record sets, fields, or analysis methods as needed for your research.**